# LSTM-DynAE Reproduction Notebook

Thin demonstration notebook for *Sequential Deep Embedded Clustering with
Dynamic Autoencoder* (MTAP-D-25-01346R2). This notebook calls functions
from the `lstm_dynae` package rather than duplicating the implementation.

**This is a demo run with a drastically reduced iteration budget** (a few
hundred iterations instead of the manuscript's 1.3 x 10^5 pretraining
iterations) so it finishes quickly on CPU/Colab. For a full, manuscript-
scale run use `scripts/pretrain.py` and `scripts/train_clustering.py`
directly with `configs/lstm_dynae.yaml`.

**AUTHOR CONFIRMATION REQUIRED items still apply** to this notebook,
exactly as documented in the README's Implementation Audit -- most
notably `clustering.max_iterations`, the exact dataset source files, and
(for EEG Eye State / Ozone Level Detection) the sequence-windowing
scheme. This notebook uses **RacketSports**, a natively sequential
dataset, to sidestep the windowing gap for a clean demonstration.

## 1. Environment setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import yaml
import matplotlib.pyplot as plt
import tensorflow as tf

from lstm_dynae.datasets import load_dataset
from lstm_dynae.preprocessing import PreprocessingConfig, augment_batch
from lstm_dynae.model import LSTMDynAE, LSTMDynAEConfig
from lstm_dynae.metrics import clustering_accuracy, normalized_mutual_information
from lstm_dynae.utils import set_global_seed

with open('../configs/lstm_dynae.yaml') as f:
    cfg = yaml.safe_load(f)

set_global_seed(cfg['seed'])
print('Seed set to', cfg['seed'], '(manuscript Section 4.3)')

## 2. Load one benchmark dataset (RacketSports)

In [ ]:
DATASET_KEY = 'RacketSports'  # natively sequential; see AUTHOR CONFIRMATION
                                # REQUIRED note above for EEG/Ozone
DATA_DIR = '../data'

prep_cfg = PreprocessingConfig(
    normalization=cfg['preprocessing']['normalization'],
    sequence_length=cfg['preprocessing']['sequence_length'],
    stride=cfg['preprocessing']['stride'],
    augment=cfg['preprocessing']['augment'],
    max_rescale_delta=cfg['preprocessing']['max_rescale_delta'],
    max_rotation_rad=cfg['preprocessing']['max_rotation_rad'],
)

# NOTE: raises DatasetNotFoundError unless you have already prepared
# data/RacketSports_X.npy / data/RacketSports_y.npy -- see data/README.md.
x, y, spec = load_dataset(DATA_DIR, DATASET_KEY, prep_cfg)
print('x shape:', x.shape, ' y shape:', y.shape, ' K =', cfg['datasets'][DATASET_KEY]['classes'])

## 3. Preprocessing (already applied by `load_dataset`; augmentation demo)

In [ ]:
sample_batch = tf.convert_to_tensor(x[:8])
augmented = augment_batch(sample_batch, prep_cfg)
print('Original batch shape:', sample_batch.shape, ' Augmented batch shape:', augmented.shape)
print('(rotation + rescale augmentation magnitudes are AUTHOR-CONFIRMATION-REQUIRED placeholders; see preprocessing.py)')

## 4. Build the LSTM Autoencoder + critic (Section 3.1)

In [ ]:
model_cfg = LSTMDynAEConfig(
    input_timesteps=x.shape[1],
    input_features=x.shape[2],
    num_clusters=cfg['datasets'][DATASET_KEY]['classes'],
    seed=cfg['seed'],
    encoder_units=tuple(cfg['model']['encoder_units']),
    decoder_units=tuple(cfg['model']['decoder_units']),
    lambda_acai=cfg['pretraining']['lambda_acai'],
    pretrain_lr=cfg['pretraining']['learning_rate'],
    pretrain_beta_1=cfg['pretraining']['first_moment_decay'],
    pretrain_beta_2=cfg['pretraining']['second_moment_decay'],
    clustering_lr=cfg['clustering']['learning_rate'],
    clustering_momentum=cfg['clustering']['momentum'],
    batch_size=min(cfg['clustering']['batch_size'], x.shape[0]),
    student_t_alpha=cfg['clustering']['student_t_alpha'],
    kappa=cfg['clustering']['kappa'],
    tau_stop=cfg['clustering']['tau_stop'],
    neighbor_pool=cfg['clustering']['neighbor_pool'],
)
model = LSTMDynAE(model_cfg)
print('Encoder units:', model_cfg.encoder_units, ' Decoder units:', model_cfg.decoder_units)
print('NOTE: critic architecture is an AUTHOR-CONFIRMATION-REQUIRED implementation choice; see critic.py')

## 5. ACAI pretraining (demo: 300 iterations, NOT the manuscript's 1.3e5)

In [ ]:
DEMO_PRETRAIN_ITERS = 300
batch_size = model_cfg.batch_size
n = x.shape[0]
ds = tf.data.Dataset.from_tensor_slices(x).shuffle(n, seed=cfg['seed']).repeat().batch(batch_size)
it = iter(ds)

pretrain_history = []
for i in range(1, DEMO_PRETRAIN_ITERS + 1):
    xb = next(it)
    if prep_cfg.augment:
        xb = augment_batch(xb, prep_cfg)
    losses = model.pretrain_step(xb)
    pretrain_history.append(float(losses['recon'].numpy()))
    if i % 50 == 0:
        print(f"iter {i}: L_fg={float(losses['L_fg']):.4f} L_C={float(losses['L_C']):.4f} recon={float(losses['recon']):.4f}")

plt.plot(pretrain_history)
plt.xlabel('iteration'); plt.ylabel('reconstruction term'); plt.title('Pretraining reconstruction loss (demo run)')
plt.show()

## 6. K-Means centroid initialization (Section 3.3.1)

In [ ]:
init_labels = model.init_centroids(x)
print('Centroids shape:', model.centroids.shape)
print('alpha_1 =', model.dynamic_loss_config.alpha_1, ' alpha_2 =', model.dynamic_loss_config.alpha_2, '(Eq. 10)')

## 7. Dynamic clustering fine-tuning (Section 3.3, Eq. 8-14)

In [ ]:
DEMO_CLUSTER_ITERS = 200  # AUTHOR CONFIRMATION REQUIRED for the real MaxItr; see README
TAU_STOP = cfg['clustering']['tau_stop']

tau_p_history, acc_history, nmi_history = [], [], []
for i in range(1, DEMO_CLUSTER_ITERS + 1):
    xb = next(it)
    if prep_cfg.augment:
        xb = augment_batch(xb, prep_cfg)
    step_out = model.clustering_train_step(xb)
    tau_p = float(step_out['tau_p'].numpy())
    tau_p_history.append(tau_p)

    if i % 20 == 0 or tau_p < TAU_STOP:
        y_pred = model.predict_clusters(x)
        acc = clustering_accuracy(y, y_pred)
        nmi = normalized_mutual_information(y, y_pred)
        acc_history.append((i, acc)); nmi_history.append((i, nmi))
        print(f"iter {i}: loss={float(step_out['loss']):.4f} tau_p={tau_p:.4f} ACC={acc:.4f} NMI={nmi:.4f}")
    if tau_p < TAU_STOP:
        print('Stopping: tau_p below tau_stop (Section 3.3.3 stopping criterion).')
        break

## 8. Final ACC / NMI (Eq. 15-16)

In [ ]:
y_pred_final = model.predict_clusters(x)
print('Final ACC:', clustering_accuracy(y, y_pred_final))
print('Final NMI:', normalized_mutual_information(y, y_pred_final))
print()
print('This DEMO run uses far fewer iterations than the manuscript and should')
print('NOT be expected to reproduce Table 3 numbers -- see run_all_datasets.py')
print('for a manuscript-scale run once MaxItr and the dataset files are confirmed.')

## 9. tau_p progression (analogous to Fig. 6/Fig. 7 of the manuscript)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tau_p_history)
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('tau_p'); axes[0].set_title('Uncertainty ratio tau_p (Eq. 11)')

if acc_history:
    xs, accs = zip(*acc_history)
    _, nmis = zip(*nmi_history)
    axes[1].plot(xs, accs, label='ACC')
    axes[1].plot(xs, nmis, label='NMI')
    axes[1].set_xlabel('iteration'); axes[1].set_title('ACC / NMI during clustering'); axes[1].legend()
plt.tight_layout(); plt.show()

## 10. Statistical significance from stored results (Section 4.9)

In [ ]:
import subprocess
out = subprocess.run(
    ['python', '../scripts/statistical_test.py', '--results', '../results/clustering_results.csv'],
    capture_output=True, text=True,
)
print(out.stdout)
if out.returncode != 0:
    print(out.stderr)